# Notebook 07 — Tool Use Basics

Define tools with JSON schemas, handle `tool_use` responses, and feed results back.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from utils.anthropic_client import client, FAST_MODEL


## Define a tool

In [ ]:
import math

TOOLS = [
    {
        "name": "calculate",
        "description": "Evaluate a Python math expression.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "e.g. 'sqrt(144)' or '2**10'"}
            },
            "required": ["expression"],
        },
    }
]

def run_tool(name, inputs):
    if name == "calculate":
        safe = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
        try:
            return str(eval(inputs["expression"], {"__builtins__": {}}, safe))
        except Exception as e:
            return f"Error: {e}"


## One-shot tool call

In [ ]:
def agent_step(messages):
    r = client.messages.create(model=FAST_MODEL, max_tokens=256, tools=TOOLS, messages=messages)
    messages.append({"role": "assistant", "content": r.content})
    if r.stop_reason == "tool_use":
        tool_results = []
        for block in r.content:
            if block.type == "tool_use":
                result = run_tool(block.name, block.input)
                print(f"  Tool {block.name}({block.input}) -> {result}")
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
        messages.append({"role": "user", "content": tool_results})
        # Get final answer
        r2 = client.messages.create(model=FAST_MODEL, max_tokens=256, tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": r2.content})
        return r2.content[0].text
    return r.content[0].text

messages = [{"role": "user", "content": "What is the square root of 1764?"}]
answer = agent_step(messages)
print("Answer:", answer)


## Exercise
Add a second tool `word_count` and test it.

In [ ]:
# Your code here
